## 2. Limpeza e Pré-processamento dos Dados

### 2.1 Tratamento de Valores Ausentes
**Justificativa da Estratégia:**
* **Remoção de Colunas:** A coluna `cancellation_reason` será completamente removida do dataset. Como ela apresenta cerca de 98% de valores nulos (já identificados na análise exploratória), qualquer técnica de imputação (como preencher com "Desconhecido") criaria um viés massivo no modelo.
* **Remoção de Linhas (Dropna):** As variáveis `actual_delivery_time_minutes`, `late_delivery` e `tip_amount` possuem nulos correspondentes aos pedidos que foram **cancelados** antes da entrega. Como o nosso objetivo futuro (Target) é prever o tempo de entrega, manter linhas de pedidos que nunca chegaram ao cliente prejudicaria o aprendizado do algoritmo. Portanto, a exclusão dessas linhas específicas é a estratégia mais coerente com a regra de negócio.

In [8]:
# 1. Importando bibliotecas e carregando os dados brutos
import pandas as pd
import numpy as np

# Carregando o dataset original (intacto)
caminho_raw = '../data/raw/food_delivery_orders_dataset.csv'
df_bruto = pd.read_csv(caminho_raw)

print(f"Tamanho original: {df_bruto.shape[0]} linhas e {df_bruto.shape[1]} colunas.")

Tamanho original: 50000 linhas e 37 colunas.


In [9]:
# Criamos uma cópia para não alterar o df_bruto na memória
df_limpo = df_bruto.copy()

# 2.1 Removendo a coluna 'cancellation_reason' (98% nulos)
df_limpo = df_limpo.drop(columns=['cancellation_reason'])

# 2.2 Removendo as linhas (pedidos) cancelados (que geraram os outros nulos)
df_limpo = df_limpo.dropna()

print(f"Tamanho após limpeza: {df_limpo.shape[0]} linhas e {df_limpo.shape[1]} colunas.")

# 2.3 Salvando o novo dataset na pasta 'processed'
caminho_processed = '../data/processed/dataset_clean.csv'
df_limpo.to_csv(caminho_processed, index=False)

Tamanho após limpeza: 49132 linhas e 36 colunas.


### 2.2 Tratamento de Outliers
**Justificativa da Estratégia (Capping/Winsorization):**
Para a variável de tempo real de entrega (`actual_delivery_time_minutes`), optamos pela **limitação (capping)** no teto de 120 minutos, em vez de remover as linhas. 
* **Por que não remover?** Tempos de entrega muito altos (ex: tempestades, acidentes) são eventos reais no negócio de delivery. Remover essas linhas faria o modelo subestimar entregas difíceis.
* **Por que não transformar (Log)?** A transformação logarítmica mudaria a escala do Target, dificultando a interpretação direta dos resultados em minutos na saída do modelo. Ao limitar em 2 horas (120 min), estabilizamos a variância sem perder a informação de que a entrega foi severamente atrasada.

In [10]:
# 3. Tratamento de Outliers (Capping / Limitação)
# Definimos um limite máximo realista para o tempo de entrega (ex: 120 minutos ou 2 horas)
limite_maximo = 120

# A função np.where funciona como um "SE" do Excel:
# SE o tempo for maior que 120, vira 120. SENÃO, mantém o valor original.
df_limpo['actual_delivery_time_minutes'] = np.where(
    df_limpo['actual_delivery_time_minutes'] > limite_maximo, 
    limite_maximo, 
    df_limpo['actual_delivery_time_minutes']
)

print("Outliers de tempo de entrega limitados a 120 minutos com sucesso!")

Outliers de tempo de entrega limitados a 120 minutos com sucesso!


In [11]:
# 4. Verificação de valores negativos inválidos
# Pegamos apenas as colunas que têm números
colunas_numericas = df_limpo.select_dtypes(include=['int64', 'float64'])

# Criamos uma checagem de quantos valores são menores que zero
valores_negativos = (colunas_numericas < 0).sum()

# Mostramos apenas as colunas que têm algum valor negativo
print("Colunas com valores negativos:")
print(valores_negativos[valores_negativos > 0])

Colunas com valores negativos:
Series([], dtype: int64)


In [12]:
# 5. Tratamento de valores negativos (Anomalias)
# Vamos usar a coluna 'distance_km' como exemplo.
# Calculamos a mediana apenas considerando os valores válidos (maiores ou iguais a zero)
mediana_distancia = df_limpo.loc[df_limpo['distance_km'] >= 0, 'distance_km'].median()

# Substituímos os valores menores que zero pela mediana calculada
df_limpo['distance_km'] = np.where(
    df_limpo['distance_km'] < 0, 
    mediana_distancia, 
    df_limpo['distance_km']
)

print(f"Possíveis valores negativos corrigidos para a mediana: {mediana_distancia:.2f} km")

Possíveis valores negativos corrigidos para a mediana: 4.10 km


### 2.3 Codificação de Variáveis Categóricas
**Explicação das Técnicas:**
* **Label Encoding:** Atribui um número inteiro sequencial para cada categoria (0, 1, 2...). É adequado **apenas** para variáveis ordinais (onde existe hierarquia, como "Baixo", "Médio", "Alto"), pois os algoritmos podem interpretar que o valor 2 é "maior" ou "duas vezes mais importante" que 1.
* **One-Hot Encoding:** Cria colunas binárias (0 ou 1) para cada categoria. É a técnica ideal para variáveis nominais (sem hierarquia).

**Estratégia Adotada:** 
Foi utilizado o **One-Hot Encoding** (`pd.get_dummies`) para variáveis como `weather` e `customer_type`, pois não há relação matemática de grandeza entre "Chuva" e "Ensolarado". O parâmetro `drop_first=True` foi aplicado para evitar a armadilha da variável dummy (multicolinearidade perfeita).

In [13]:
# 6. Codificação de Variáveis Categóricas (One-Hot Encoding)
# Escolhendo colunas de texto que são importantes para as entregas
colunas_para_codificar = ['weather', 'traffic_level', 'customer_type', 'day_of_week']

# O get_dummies transforma os textos em colunas de 0 (Falso) e 1 (Verdadeiro)
# drop_first=True remove uma das opções para evitar informações repetidas (ex: se não é dia, é noite)
df_limpo = pd.get_dummies(df_limpo, columns=colunas_para_codificar, drop_first=True)

# Convertendo os booleanos (True/False) que o pandas gera para números inteiros (1 e 0)
for col in df_limpo.columns:
    if df_limpo[col].dtype == bool:
        df_limpo[col] = df_limpo[col].astype(int)

print(f"Novo tamanho do dataset: {df_limpo.shape[0]} linhas e {df_limpo.shape[1]} colunas.")

Novo tamanho do dataset: 49132 linhas e 45 colunas.


### 2.4 Padronização e Normalização
**Avaliação da Necessidade:**
Algoritmos baseados em cálculo de distância (como KNN e SVM) ou em gradiente descendente (como Redes Neurais e Regressão Logística) são extremamente sensíveis à escala dos dados. Sem tratamento, variáveis com valores altos (como `order_total`) dominariam o modelo em detrimento de variáveis menores (como `distance_km`).

**Justificativa da Técnica:**
Optamos pela **Normalização (Min-Max Scaling)** ao invés da Padronização (Z-score). O Min-Max Scaler comprime todos os dados numéricos no intervalo de `[0, 1]`. Isso foi escolhido porque muitas das nossas variáveis não seguem uma distribuição Normal/Gaussiana perfeita, o que torna o Min-Max mais seguro e robusto para preservar a forma original da distribuição dos dados.

In [14]:
import joblib
from sklearn.preprocessing import MinMaxScaler

# 1. Selecionando colunas numéricas elegíveis para escala
colunas_numericas = df_limpo.select_dtypes(include=['int64', 'float64']).columns
colunas_para_escalar = [col for col in colunas_numericas if df_limpo[col].max() > 1]

# 2. Criando e salvando o Scaler exclusivo para o Target (tempo de entrega)
# Isso permite reverter a escala na hora do gráfico!
scaler_y = MinMaxScaler()
df_limpo['actual_delivery_time_minutes'] = scaler_y.fit_transform(df_limpo[['actual_delivery_time_minutes']])

# Salvando o objeto scaler do target
joblib.dump(scaler_y, '../models/scaler_y.pkl')

# 3. Escalando as demais variáveis de entrada (Features)
colunas_features = [col for col in colunas_para_escalar if col != 'actual_delivery_time_minutes']
scaler_X = MinMaxScaler()
df_limpo[colunas_features] = scaler_X.fit_transform(df_limpo[colunas_features])

# Salvando o objeto scaler das features
joblib.dump(scaler_X, '../models/scaler_X.pkl')

# 4. Salvando o dataset processado
caminho_final = '../data/processed/dataset_preprocessed.csv'
df_limpo.to_csv(caminho_final, index=False)

print("Normalização concluída e objetos Scaler salvos com sucesso em ../models/!")

Normalização concluída e objetos Scaler salvos com sucesso em ../models/!
